In [ ]:
print('===== NATIVE EMBED BWD BITEXACT TEST (C helper injected) =====')
print('===== NATIVE SOFTMAX CB V2 - CTYPES FIXED =====')
print('===== NATIVE SOFTMAX CAUSAL BWD BITEXACT TEST =====')
print('===== NATIVE SOFTMAX CAUSAL FWD BITEXACT TEST =====')
print('===== NATIVE MERGEHBHEAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (has BiasAddRows) + .tkv-generated C .so
import subprocess, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/embed_bwd_cpu.cpu.c'; so=ex+'/embed_bwd_cpu.so'
# inject exact EmbedBwdFused C from tkv_bwd_ops.c (calloc/starts/cursor/pos)
cc_path=ex+'/embed_bwd_cpu.cpu.c'
txt=open(cc_path).read()
eb=r"""
#include <stdlib.h>
#include <string.h>
static inline void tkv_embed_bwd(const long long* idx, const float* gy, float* gw,
                                 int rows, int d, int n) {
    memset(gw, 0, (size_t)rows * d * sizeof(float));
    if (n == 0) return;
    int* counts=(int*)calloc((size_t)rows,sizeof(int));
    int* starts=(int*)malloc((size_t)rows*sizeof(int));
    int* cursor=(int*)malloc((size_t)rows*sizeof(int));
    int* pos=(int*)malloc((size_t)n*sizeof(int));
    if (!counts||!starts||!cursor||!pos){free(counts);free(starts);free(cursor);free(pos);return;}
    for (int r=0;r<rows;r++) counts[r]=0;
    for (int t=0;t<n;t++){int r=(int)idx[t]; if(r<0||r>=rows){free(counts);free(starts);free(cursor);free(pos);return;} counts[r]++;}
    int s=0; for(int r=0;r<rows;r++){starts[r]=s;s+=counts[r];}
    for(int r=0;r<rows;r++) cursor[r]=starts[r];
    for(int t=0;t<n;t++){int r=(int)idx[t]; pos[cursor[r]]=t; cursor[r]++;}
    for(int q=0;q<n;q++){int t=pos[q];int r=(int)idx[t];float* gr=gw+(long long)r*d;const float* gyrow=gy+(long long)t*d;for(int j=0;j<d;j++)gr[j]+=gyrow[j];}
    free(counts);free(starts);free(cursor);free(pos);
}
"""
txt=txt.replace('#include <math.h>', '#include <math.h>\n'+eb)
open(cc_path,'w').write(txt)
print('embed C injected =', 'tkv_embed_bwd' in open(cc_path).read())
p2=subprocess.run('gcc -O2 -mavx2 -mfma -ffp-contract=off -fno-fast-math -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare EmbedBwdFused
import ctypes, numpy as np
F32=ctypes.c_float; I64I=ctypes.c_longlong; I32=ctypes.c_int
args=[ctypes.POINTER(I64I),I32,I32,I32,ctypes.POINTER(F32),I32,ctypes.POINTER(F32),I32]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').EmbedBwdFused
native.argtypes=args; native.restype=I32
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/embed_bwd_cpu.so').EmbedBwdFused
tkv.argtypes=args; tkv.restype=I32
rng=np.random.default_rng(7)
for rows,d,n in [(16,8,32),(64,16,100),(128,32,500),(7,8,20)]:
    idx=rng.integers(0,rows,n).astype(np.int64); gy=rng.standard_normal(n*d).astype(np.float32)
    ga=np.zeros(rows*d,np.float32); gc=np.zeros(rows*d,np.float32)
    ia=idx.ctypes.data_as(ctypes.POINTER(I64I)); ya=gy.ctypes.data_as(ctypes.POINTER(F32))
    native(ia,n,rows,d,ya,gy.size,ga.ctypes.data_as(ctypes.POINTER(F32)),ga.size)
    tkv(ia,n,rows,d,ya,gy.size,gc.ctypes.data_as(ctypes.POINTER(F32)),gc.size)
    diff=np.count_nonzero(ga.view(np.uint32)!=gc.view(np.uint32))
    print('rows=%d d=%d n=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(rows,d,n,diff==0,diff,ga.size,np.abs(ga-gc).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> BiasAddRows can be replaced by .tkv-generated C.')